# Week 2 Tue: pandas DataFrames, indexing, joins, and missing values

Estimated time: 6 hours

## Why this matters
pandas DataFrames, indexing, joins, and missing values is part of real quant work inside data and linear algebra: numpy, pandas, visualization, diversification, and sql basics research, trading, or risk workflows.

## Session plan
- Session 1 (45 min): Learn DataFrame structure and indexing.
- Session 2 (55 min): Clean missing values and basic data problems.
- Session 3 (55 min): Join and reshape simple market tables.
- Session 4 (55 min): Code drills and small cleaning exercises.
- Session 5 (30 min): Reflection and interview recap.

## Concept notes
### DataFrames as structured market tables
A DataFrame is like a spreadsheet with labeled rows and columns.

Technical note: pandas DataFrames provide indexing, grouping, joining, missing-value handling, and time-series functionality used heavily in research workflows.

Finance example: You might store dates, prices, returns, sectors, and volumes in one table and transform it before modeling.

### Missing values and bad data
Real market data often has gaps, duplicate records, or inconsistent labels.

Technical note: Cleaning decisions matter because dropped rows, forward fills, or wrong joins can change backtest outputs.

Finance example: A missing price on one date can distort return calculations unless handled explicitly.

### Joins and alignment
A join combines tables using a common key such as date or ticker.

Technical note: Misaligned joins are a common source of subtle leakage and incorrect analysis in finance data pipelines.

Finance example: Joining returns with a factor table on the wrong date can accidentally use future information.

## Continuity prompt
- What from yesterday's topic still needs reinforcement?
- What should tomorrow build on from today?

## Real-world dataset for today's labs
Use `curriculum/datasets/real_market_prices.csv` as your default market panel (SPY, QQQ, TLT, GLD).


## Code Lab 1: Build and inspect a price table

In [1]:
import pandas as pd

prices = pd.DataFrame(
    {
        "date": pd.date_range("2026-01-01", periods=5, freq="D"),
        "asset_a": [100, 101, None, 103, 104],
        "asset_b": [50, 49, 50, 51, 52],
    }
).set_index("date")
print(prices)
print("\nMissing values:")
print(prices.isna().sum())


            asset_a  asset_b
date                        
2026-01-01    100.0       50
2026-01-02    101.0       49
2026-01-03      NaN       50
2026-01-04    103.0       51
2026-01-05    104.0       52

Missing values:
asset_a    1
asset_b    0
dtype: int64


## Code Lab 2: Clean and compute returns

In [2]:
cleaned = prices.ffill()
returns = cleaned.pct_change().dropna()
print(cleaned)
print("\nReturns:")
print(returns.round(4))


            asset_a  asset_b
date                        
2026-01-01    100.0       50
2026-01-02    101.0       49
2026-01-03    101.0       50
2026-01-04    103.0       51
2026-01-05    104.0       52

Returns:
            asset_a  asset_b
date                        
2026-01-02   0.0100  -0.0200
2026-01-03   0.0000   0.0204
2026-01-04   0.0198   0.0200
2026-01-05   0.0097   0.0196


## Code Lab 3: Join with simple metadata

In [3]:
metadata = pd.DataFrame(
    {
        "ticker": ["asset_a", "asset_b"],
        "sector": ["equity", "commodity_proxy"],
    }
)
melted = cleaned.reset_index().melt(id_vars="date", var_name="ticker", value_name="price")
merged = melted.merge(metadata, on="ticker", how="left")
print(merged.head())


        date   ticker  price  sector
0 2026-01-01  asset_a  100.0  equity
1 2026-01-02  asset_a  101.0  equity
2 2026-01-03  asset_a  101.0  equity
3 2026-01-04  asset_a  103.0  equity
4 2026-01-05  asset_a  104.0  equity


## Practice recap
- Why is missing data dangerous in return calculations?
  Suggested answer: Because missing values can create false jumps, incorrect percentage changes, or unintended row drops if handled casually.
- What is one common join mistake in finance data?
  Suggested answer: Joining on the wrong date key and accidentally aligning today's feature with tomorrow's outcome.
- Why do labels matter in pandas?
  Suggested answer: Because labeled indices and columns make transformations safer and easier to interpret than raw position-only arrays.

## Interview drill
- Q: What is the difference between a DataFrame and a NumPy array in practice?
  A: A DataFrame has labels and richer data-wrangling features, while a NumPy array is more lightweight and numerically focused.

## 6-Hour Completion Roadmap

Use this minimum structure to turn today's notebook into a serious study session:

- Phase 1 (60 min): Read concept notes and rewrite the core idea from memory.
- Phase 2 (60 min): Complete and extend all code labs with at least one variation.
- Phase 3 (60 min): Do formula retrieval, scenario analysis, and error-log updates.
- Phase 4 (60 min): Practice interview responses aloud.
- Phase 5 (60 min): Implement one robustness check.
- Phase 6 (60 min): Write a concise memo with limitations and next steps.

## Formula Rewrite Drill

In [4]:
import pandas as pd

formula_drill = pd.DataFrame(
    [{'formula': 'MR=\\frac{\\#\\text{missing cells}}{\\#\\text{total cells}}', 'from_memory': '', 'term_explanations': ''}, {'formula': 'Rows_{out}=Rows_A\\bowtie Rows_B', 'from_memory': '', 'term_explanations': ''}, {'formula': 'Coverage_i=\\frac{\\#dates_i}{\\#dates_{max}}', 'from_memory': '', 'term_explanations': ''}]
)
print(formula_drill)


                                             formula from_memory  \
0  MR=\frac{\#\text{missing cells}}{\#\text{total...               
1                    Rows_{out}=Rows_A\bowtie Rows_B               
2         Coverage_i=\frac{\#dates_i}{\#dates_{max}}               

  term_explanations  
0                    
1                    
2                    


## Real Market Data Lab (Topic-Aligned)

        Use today's topic (pandas DataFrames, indexing, joins, and missing values) with one concrete dataset and one measurable output.

        - Use yfinance first for SPY, QQQ, TLT, and GLD when internet is available.
- If available, validate against a Robinhood-style export CSV for consistency checks.
- Fall back to curriculum/datasets/real_market_prices.csv for reproducible runs.
- Design one topic-specific analysis for pandas dataframes, indexing, joins, and missing values instead of reusing generic volatility-only metrics.
- Document one implementation risk and one robustness check before finalizing conclusions.

In [5]:
TASK = "Load market data and compute a correlation/covariance diagnostic tied to today's topic."
print('Lab task:', TASK)

from pathlib import Path
import pandas as pd

market = pd.read_csv(Path("curriculum/datasets/real_market_prices.csv"), parse_dates=["date"])
prices = market.pivot(index="date", columns="symbol", values="close").dropna()
returns = prices.pct_change().dropna()
print(returns.corr().round(3))
print("\nCovariance:")
print(returns.cov().round(6))


Lab task: Load market data and compute a correlation/covariance diagnostic tied to today's topic.
symbol    GLD    QQQ    SPY    TLT
symbol                            
GLD     1.000  0.123  0.131  0.210
QQQ     0.123  1.000  0.947  0.081
SPY     0.131  0.947  1.000  0.068
TLT     0.210  0.081  0.068  1.000

Covariance:
symbol       GLD       QQQ       SPY       TLT
symbol                                        
GLD     0.000124  0.000019  0.000016  0.000023
QQQ     0.000019  0.000200  0.000143  0.000011
SPY     0.000016  0.000143  0.000114  0.000007
TLT     0.000023  0.000011  0.000007  0.000099


## Real-World Takeaway Prompt

Write 5-8 lines for today's topic (pandas DataFrames, indexing, joins, and missing values):

1. Which symbol looked most volatile and why that matters.
2. Which pair looked most diversifying.
3. One realistic portfolio/risk decision you could make from this table.

## Interview Question + Python Solution Drill

Use this format for interview preparation:

1. State the question clearly.
2. Explain your approach in plain language.
3. Write Python code on relevant data.
4. Interpret one risk caveat in words.

**Drill prompt**
- Load market data and compute a correlation/covariance diagnostic tied to today's topic.

In [6]:
TASK = "Load market data and compute a correlation/covariance diagnostic tied to today's topic."
print('Interview drill task:', TASK)

from pathlib import Path
import pandas as pd

market = pd.read_csv(Path("curriculum/datasets/real_market_prices.csv"), parse_dates=["date"])
prices = market.pivot(index="date", columns="symbol", values="close").dropna()
returns = prices.pct_change().dropna()
print(returns.corr().round(3))
print("\nCovariance:")
print(returns.cov().round(6))


Interview drill task: Load market data and compute a correlation/covariance diagnostic tied to today's topic.
symbol    GLD    QQQ    SPY    TLT
symbol                            
GLD     1.000  0.123  0.131  0.210
QQQ     0.123  1.000  0.947  0.081
SPY     0.131  0.947  1.000  0.068
TLT     0.210  0.081  0.068  1.000

Covariance:
symbol       GLD       QQQ       SPY       TLT
symbol                                        
GLD     0.000124  0.000019  0.000016  0.000023
QQQ     0.000019  0.000200  0.000143  0.000011
SPY     0.000016  0.000143  0.000114  0.000007
TLT     0.000023  0.000011  0.000007  0.000099


## Scenario Analysis Drill

In [7]:
import pandas as pd

scenarios = pd.DataFrame(
    [
        {"scenario": "base_case", "assumption": "", "expected_effect": ""},
        {"scenario": "bull_case", "assumption": "", "expected_effect": ""},
        {"scenario": "stress_case", "assumption": "", "expected_effect": ""},
    ]
)
print("Topic:", 'pandas DataFrames, indexing, joins, and missing values')
print(scenarios)


Topic: pandas DataFrames, indexing, joins, and missing values
      scenario assumption expected_effect
0    base_case                           
1    bull_case                           
2  stress_case                           


## Closed-Book Retrieval and Error Log

In [8]:
import pandas as pd

retrieval_scorecard = pd.DataFrame(
    [
        {"prompt": "Explain today's concept in 3 lines", "score_0_to_5": None, "notes": ""},
        {"prompt": "Write one key formula without notes", "score_0_to_5": None, "notes": ""},
        {"prompt": "Give one realistic failure mode", "score_0_to_5": None, "notes": ""},
        {"prompt": "Connect to one trading/risk decision", "score_0_to_5": None, "notes": ""},
    ]
)

error_log = pd.DataFrame(
    [
        {
            "concept": "",
            "mistake": "",
            "correction": "",
            "next_review_date": "",
        }
    ]
)
print("Retrieval scorecard template:")
print(retrieval_scorecard)
print("\nError log template:")
print(error_log)


Retrieval scorecard template:
                                 prompt score_0_to_5 notes
0    Explain today's concept in 3 lines         None      
1   Write one key formula without notes         None      
2       Give one realistic failure mode         None      
3  Connect to one trading/risk decision         None      

Error log template:
  concept mistake correction next_review_date
0                                            


## Final 30-Minute Deliverable

Write a short memo (150-250 words) with this structure:

1. Core idea of pandas DataFrames, indexing, joins, and missing values in plain language.
2. One technical detail or formula and why it matters.
3. One practical quant use case.
4. One limitation or failure mode and how you would detect it.